# 01 — CICIDS2017 Exploratory Data Analysis

**Prerequisite:** `python scripts/01_prepare_data.py`

This notebook supports **RQ1/RQ2** (data understanding + class balance).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
ROOT = Path("..").resolve()
processed = ROOT / "data" / "processed"
assert (processed / "train.parquet").exists(), "Run scripts/01_prepare_data.py first"

train = pd.read_parquet(processed / "train.parquet")
val = pd.read_parquet(processed / "val.parquet")
test = pd.read_parquet(processed / "test.parquet")
summary = json.loads((processed / "summary.json").read_text())
summary

## Split sizes & label distribution

In [ ]:
print({
    "train": len(train),
    "val": len(val),
    "test": len(test),
    "attack_rate_train": float(train["is_attack"].mean()),
})

counts = train["Label"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="bar", ax=axes[0], color="#3ec7c2")
axes[0].set_title("Train label counts")
axes[0].tick_params(axis="x", rotation=45)

pct = train["Label"].value_counts(normalize=True) * 100
pct.plot(kind="bar", ax=axes[1], color="#e4a54a")
axes[1].set_title("Train label %")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
pd.DataFrame({"count": counts, "percent": pct.round(2)})

## Benign vs attack balance

In [ ]:
balance = train["is_attack"].value_counts().rename({0: "BENIGN", 1: "ATTACK"})
ax = balance.plot(kind="bar", color=["#5ecf8a", "#e35d6a"], figsize=(5, 3))
ax.set_title("Binary class balance (train)")
ax.set_ylabel("Flows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
balance

## Feature overview

Numeric summary for a sample of flow features (helps motivate scaling + selection).

In [ ]:
feature_cols = [c for c in train.columns if c not in ("Label", "is_attack")]
print("n_features:", len(feature_cols))
desc = train[feature_cols].describe().T
desc["missing"] = train[feature_cols].isna().sum()
desc.sort_values("std", ascending=False).head(20)

## Correlation snapshot (attack vs selected rates)

Uses a stratified subsample for speed.

In [ ]:
candidates = [
    c for c in [
        "Flow Duration", "Flow Bytes/s", "Flow Packets/s",
        "Total Fwd Packets", "Total Backward Packets",
        "Packet Length Mean", "Fwd IAT Mean", "Bwd IAT Mean",
        "Average Packet Size", "Init_Win_bytes_forward",
    ] if c in train.columns
]
sample = train.groupby("is_attack", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 5000), random_state=42)
)
corr = sample[candidates + ["is_attack"]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Correlation heatmap (subsample)")
plt.tight_layout()
plt.show()
corr["is_attack"].drop("is_attack").abs().sort_values(ascending=False)

## Link to trained model metrics

In [ ]:
report_path = ROOT / "models" / "trained_models" / "training_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("Best binary:", report["binary"]["best"])
    print("Binary test F1/Recall:", report["binary"]["test"]["f1"], report["binary"]["test"]["recall"])
    print("Best multiclass:", report["multiclass"]["best"])
    print("Multiclass weighted F1:", report["multiclass"]["test"]["f1_weighted"])
else:
    print("Train models first: python scripts/02_train_models.py")